# TM Forum Open APIs & ODA Components RAG — Quickstart Notebook

**Open Telco RAG**

This notebook walks through the Open Telco RAG pipeline for the
**TM Forum** corpus — TMF Open APIs (TMF620–TMF936) and ODA Component
specifications from the [tmforum-rand/tmf-otel](https://github.com/tmforum-rand/tmf-otel) repository.

By the end of this notebook you will have:
- Ingested TM Forum Open API and ODA Component specs into a local ChromaDB
- Searched across API endpoints, data models, and component definitions
- Asked natural language questions about TM Forum specifications
- Evaluated retrieval quality with the Open Telco RAG evaluation framework

### Prerequisites

```bash
git clone https://github.com/Znbne/telco-retrieve
cd telco-retrieve
pip install -e .
```

Set your API keys in a `.env` file at the project root:
```
# For answer generation (choose one or both)
ANTHROPIC_API_KEY=sk-ant-...        # Claude models direct
OPENROUTER_API_KEY=sk-or-...        # Any model via OpenRouter
```

Download TM Forum specifications:
```bash
git clone https://github.com/tmforum-rand/tmf-otel data/tmforum
```


## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from collections import Counter

load_dotenv()

DATA_DIR   = Path("data/tmforum")
DB_PATH    = "chroma_db"
MODEL_KEY  = "minilm"
COLLECTION = f"tmforum_{MODEL_KEY}"

print(f"Data directory : {DATA_DIR}")
print(f"ChromaDB path  : {DB_PATH}")
print(f"Collection     : {COLLECTION}")


## 2. Check available content

The TM Forum corpus has two types of specifications:
- **TMF Open APIs** — OpenAPI specifications for service, resource, product, billing, and party management (JSON and YAML)
- **ODA Components** — Component specifications defining functional boundaries and API dependencies (YAML)


In [ ]:
if DATA_DIR.exists():
    api_yaml  = list((DATA_DIR / "apis").glob("*.yaml"))
    api_json  = list((DATA_DIR / "apis").glob("*.json"))
    comp_yaml = list((DATA_DIR / "components").glob("*.yaml"))
    print(f"API specs (OAS YAML) : {len(api_yaml)}")
    print(f"API specs (JSON)     : {len(api_json)}")
    print(f"ODA Components       : {len(comp_yaml)}")
    print(f"Total files          : {len(api_yaml) + len(api_json) + len(comp_yaml)}")
    print("\nKey APIs:")
    for f in sorted(api_yaml)[:5]:
        print(f"  {f.name}")
else:
    print(f"[INFO] {DATA_DIR} not found.")
    print("Clone: git clone https://github.com/tmforum-rand/tmf-otel data/tmforum")


## 3. Ingest into ChromaDB

The TM Forum ingest pipeline:
1. Loads OpenAPI specs (YAML/JSON) and converts endpoints, parameters, and data models to prose
2. Loads ODA Component specs and extracts exposed/dependent APIs and component metadata
3. Chunks the prose with `chunk_size=1024`, `overlap=200`
4. Embeds with the selected model and stores in ChromaDB


In [ ]:
from open_telco_rag.sources.tmforum import TMForumSource
from open_telco_rag.embedders.sentense_transformers import SentenceTransformerEmbedder
from open_telco_rag.stores.chroma import ChromaStore
from open_telco_rag.core.pipeline import ingest

source   = TMForumSource()
embedder = SentenceTransformerEmbedder(model_key=MODEL_KEY)
store    = ChromaStore(path=DB_PATH, collection_name=COLLECTION)

stats = ingest(
    source     = source,
    store      = store,
    embedder   = embedder,
    data_dir   = str(DATA_DIR),
    chunk_size = 1024,
    overlap    = 200,
    resume     = True,
)

print(f"Files processed : {stats.files_processed}")
print(f"Chunks stored   : {stats.chunks_stored}")
print(f"Skipped         : {stats.skipped_files}")
print(f"Errors          : {stats.error_files}")


## 4. Explore the collection

TM Forum metadata includes the TMF number, API name, doc type, and version.


In [ ]:
import chromadb

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(COLLECTION)
sample     = collection.get(limit=2000, include=["documents", "metadatas"])

tmf_counts  = Counter(m.get("tmf_number", "?")  for m in sample["metadatas"])
type_counts = Counter(m.get("doc_type", "?")     for m in sample["metadatas"])
repo_counts = Counter(m.get("repo_type", "?")    for m in sample["metadatas"])

print(f"Total chunks: {collection.count():,}")
print(f"\nTop TMF APIs by chunk count:")
for tmf, count in sorted(tmf_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {tmf:<12} {count:>5} chunks")
print(f"\nDoc types:")
for dtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {dtype:<30} {count:>5}")


## 5. Semantic Search

TM Forum chunks contain API endpoint definitions, data model descriptions,
and ODA component dependency specifications.


In [ ]:
from open_telco_rag.core.pipeline import search

QUERY = "What APIs does the TMF620 Product Catalog Management expose?"
TOP_K = 5

results = search(
    QUERY,
    store    = store,
    embedder = embedder,
    top_k    = TOP_K,
)

print(f"Query: '{QUERY}'")
print(f"Top-{TOP_K} results:\n")
print(f"{'Rank':<5} {'Similarity':>10} {'TMF':>8} {'Source'}")
print("-" * 70)

for i, r in enumerate(results, 1):
    import math
    sim    = round(math.exp(-r.distance), 3)
    source = r.metadata.get("source", "")[:40]
    tmf    = r.metadata.get("tmf_number", "")
    print(f"{i:<5} {sim:>10.3f} {tmf:>8} {source}")

print(f"\nTop result preview:")
print(results[0].text[:400])


### 5.1 ODA Component discovery

ODA Components define which APIs they expose and depend on.
Searching component specs reveals the functional architecture of TM Forum-compliant systems.


In [ ]:
from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

QUERY2 = "What APIs does the TMFC001 Product Catalog Management ODA component expose and depend on?"

results2 = search(QUERY2, store=store, embedder=embedder, top_k=5)

print(f"Query: '{QUERY2}'\n")
for i, r in enumerate(results2, 1):
    import math
    sim  = round(math.exp(-r.distance), 3)
    tmfc = r.metadata.get("tmfc_number", r.metadata.get("tmf_number", ""))
    print(f"  {i}. [{tmfc}] sim={sim:.3f} — {r.text[:120]}...")


## 6. Question Answering

In [ ]:
QUESTION = "What APIs does the TMF620 Product Catalog Management expose?"

retrieved = search(QUESTION, store=store, embedder=embedder, top_k=5)

from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

from open_telco_rag.core.pipeline import generate
response = generate(QUESTION, retrieved, source_type="tmforum", generator=generator)

print(f"Question: {QUESTION}")
print(f"{'─'*70}")
print(f"\n{response['answer']}")
print(f"\n{'─'*70}")
print(f"Sources used: {response['sources']}")
print(f"Chunks used : {response['chunks_used']}")


### 6.1 Question bank

Try these questions across TM Forum Open APIs and ODA Components.


In [ ]:
QUESTIONS = [
    ("What APIs does the TMF620 Product Catalog Management expose?",           "TMF620"),
    ("What is the TMF921 Intent Management API and what does it define?",      "TMF921"),
    ("How does TMFC001 Product Catalog Management interact with other components?", "TMFC001"),
    ("What does the TMF641 Service Order Management API provide?",             "TMF641"),
    ("What are the mandatory parameters for creating a product order in TMF622?", "TMF622"),
    ("How does TM Forum define ODA component functional boundaries?",          "ODA"),
    ("What does the TMF632 Party Management API expose?",                      "TMF632"),
]

print(f"{'#':<3} {'Question':<65} {'Domain'}")
print("-" * 80)
for i, (q, domain) in enumerate(QUESTIONS, 1):
    q_short = q[:62] + "..." if len(q) > 62 else q
    print(f"{i:<3} {q_short:<65} {domain}")


## 7. Evaluation

Run the evaluation suite across TM Forum evaluation questions.

**Note**: metric scoring calls the LLM API (configured below).
The full run takes approximately 3–5 minutes for 10 questions.


In [ ]:
from open_telco_rag.evaluation.evaluate import evaluate_corpus
from open_telco_rag.evaluation.questions import load_questions

questions = load_questions("tmforum")
print(f"Loaded {len(questions)} evaluation questions for TM Forum")
print(f"Questions with reference answers: {sum(1 for q in questions if q.reference_answer)}")


In [ ]:
from open_telco_rag.evaluation._claude import make_llm_call

# RAG generation model (answers questions)
rag_llm_call = make_llm_call("openrouter", "meta-llama/llama-3.3-70b-instruct")

# Metrics evaluation model (scores faithfulness, relevancy, etc.)
llm_call = make_llm_call("openrouter", "anthropic/claude-sonnet-4.5")

result, baseline_result = evaluate_corpus(
    corpus       = "tmforum",
    store        = store,
    embedder     = embedder,
    questions    = questions,
    model        = MODEL_KEY,
    top_k        = 5,
    delay        = 1.0,
    llm_call     = llm_call,
    rag_llm_call = rag_llm_call,
)


In [ ]:
print(f"\nPer-question results — TM Forum ({MODEL_KEY}, top-k=5)")
print(f"{'#':<3} {'Faith':>6} {'Relev':>6} {'Sim':>6} {'Recall':>7} {'Correct':>8} {'Question'}")
print("-" * 95)

for i, qr in enumerate(result.question_results, 1):
    q_short = qr.question[:45] + "..." if len(qr.question) > 45 else qr.question
    print(
        f"{i:<3} "
        f"{qr.faithfulness:>6.2f} "
        f"{qr.answer_relevancy:>6.2f} "
        f"{qr.mean_similarity:>6.3f} "
        f"{max(0, qr.context_recall):>7.2f} "
        f"{max(0, qr.answer_correctness):>8.2f} "
        f"{q_short}"
    )

print("-" * 95)
print(
    f"{'AVG':<3} "
    f"{result.mean_faithfulness:>6.2f} "
    f"{result.mean_answer_relevancy:>6.2f} "
    f"{result.mean_similarity:>6.3f} "
    f"{max(0, result.mean_context_recall):>7.2f} "
    f"{max(0, result.mean_answer_correctness):>8.2f}"
)


## 8. What the metrics tell us

| Metric | Type | Interpretation |
|--------|------|----------------|
| **Faithfulness** | Generation | Are the claims in the answer grounded in the retrieved chunks? |
| **Answer relevancy** | Generation | Does the answer address what was asked? |
| **Mean similarity** | Retrieval | How confident is the retrieval? Higher = better chunk matches |
| **Context recall** | Retrieval+GT | Were the chunks needed to answer the question retrieved? |
| **Answer correctness** | Generation+GT | Is the answer factually correct vs the reference answer? |
